# 区分股价修复与披露篮子更新
对既有描述量做恒等分解；沿用5/20日端点，未知状态保留，不以未来收益选择定义。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.analysis.breadth_change_decomposition import decompose_breadth_changes
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/breadth_change_semantics_20261014';plan=json.loads((card/'plan.json').read_text());parent=root/plan['source_card'];s=json.loads((parent/'summary.json').read_text());qlib.init(provider_uri=s['provider_uri'],region='us');mp=parent/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==plan['source_manifest_sha256']
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=plan['source_recorder_id'],experiment_name=parent.name);entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/analysis/prices'));path=Path(rec.download_artifact('analysis/prices'));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];prices=rec.load_object('analysis/prices')
history_path=root/plan['holdings'];assert hashlib.sha256(history_path.read_bytes()).hexdigest()==plan['holdings_sha256'];h=pd.read_csv(history_path,parse_dates=['filing_date','report_date']);snapshots=h[['report_date','filing_date','accession']].drop_duplicates().sort_values(['report_date','filing_date','accession'])
w=pd.DataFrame(np.nan,index=prices.index,columns=prices.columns);versions=pd.Series(None,index=prices.index,dtype=object)
for day in prices.index:
 eligible=snapshots.loc[snapshots.filing_date.lt(day)]
 if eligible.empty:continue
 snap=eligible.iloc[-1];members=h.loc[h.accession.eq(snap.accession)].set_index('ticker').weight_pct;w.loc[day]=0.;w.loc[day,members.index]=members;versions.loc[day]=snap.accession
ret5=prices/prices.shift(5)-1.;low20=prices.rolling(20,min_periods=20).min()
indicators={'positive_return5':ret5.gt(0).astype(float).where(ret5.notna()),'new_low20':prices.le(low20).astype(float).where(low20.notna()&prices.notna())}
results={};rows=[];events=[]
for name,indicator in indicators.items():
 out=decompose_breadth_changes(indicator,w);out['basket_version']=versions;out['basket_changed']=versions.ne(versions.shift(1))&versions.notna()&versions.shift(1).notna();out['price_change5']=out.price_component.rolling(5,min_periods=5).sum();out['basket_change5']=out.basket_component.rolling(5,min_periods=5).sum()
 ready=out.total_change.notna();assert out.loc[ready,'reconciliation_error'].abs().max()<1e-12
 assert out.loc[ready&~out.basket_changed,'basket_component'].abs().max()<1e-12
 np.testing.assert_allclose((out.price_change5+out.basket_change5).dropna(),(out.current-out.current.shift(5)).reindex((out.price_change5+out.basket_change5).dropna().index),rtol=1e-10,atol=1e-12)
 results[name]=out
 for window,start,end in [('all','2019-11-26',plan['as_of']),('2023Q4','2023-10-01','2023-12-31')]:
  part=out.loc[start:end];valid=part.total_change.notna();changed=part.basket_changed&valid
  rows.append({'feature':name,'window':window,'known_levels':int(part.current.notna().sum()),'decomposable_changes':int(valid.sum()),'basket_change_dates':int(part.basket_changed.sum()),'decomposable_basket_changes':int(changed.sum()),'max_absolute_basket_change_pp':float(part.basket_component.abs().max()*100),'sum_absolute_price_changes_pp':float(part.price_component.abs().sum()*100),'sum_absolute_basket_changes_pp':float(part.basket_component.abs().sum()*100)})
 events.append(out.loc[out.basket_changed,['current','price_component','basket_component','total_change']].assign(feature=name).reset_index())
summary={'experiment':card.name,'decision':'descriptive_breadth_price_and_basket_changes_separated','new_fits':0,'new_backtests':0,'parameter_searches':0,'outcome_correlations':0,'objective_achieved':False,'provider_uri':s['provider_uri'],'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Reported-basket descriptive decomposition; no new alpha claim or reopening of closed four-feature breadth experiment. Entire historical calendar and unknown changes retained.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='no_fit_basket_change_decomposition',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,weights=w)
 rec.save_objects(artifact_path='analysis',daily=results,summary=summary,metrics=pd.DataFrame(rows))
 rec.save_objects(artifact_path='sources',decomposition=(root/'src/analysis/breadth_change_decomposition.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(rows).to_csv(card/'decomposition_summary.csv',index=False);pd.concat(events).to_csv(card/'disclosure_events.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':s['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));print(pd.DataFrame(rows).to_string(index=False))


# 广度变化的含义：区分价格移动与披露篮子更新

本轮0拟合、0回测、0未来收益相关检验、0参数搜索。使用已衔接补充来源的披露篮子和真实价格，固定5日上涨占比、20日新低占比两项描述量；不重开10月5日关闭的四特征广度实验。

恒等式：B(x_t,w_t)-B(x_{t-1},w_{t-1})=[B(x_t,w_{t-1})-B(x_{t-1},w_{t-1})]+[B(x_t,w_t)-B(x_t,w_{t-1})]。左侧为总变化，第一项保留旧篮子只更新价格指标，第二项保留当日指标只更新篮子/权重。篮子必须在此前日期已披露，不能回填报告日。五日分量仅为固定逐日分量求和，不估计任何权重。

两项指标各有1027个已知水平、1025个可分解日变化；26次披露版本更新中15次可分解。其余日期和更新事件保留，不因旧篮子当日缺价而删除；有时当前水平已知但变化分解仍未知。合成检查验证：价格指标不变、只换权重时，变化全部归篮子；旧成员当日未知时不假装价格项为0。真实数据恒等式及五日加总全部通过。

全区间单日最大绝对篮子效应：上涨占比6.1993个百分点，新低占比3.0679个百分点。此处百分点是广度指标变化，不是投资收益。不能将披露更新日的全部变化命名为股价修复。

2023Q4的63日全部可分解，只有一次篮子更新；上涨占比篮子效应1.01995个百分点，新低占比效应0。该季度绝大多数观察到的广度变化来自价格指标变化，不能用“只是换了篮子”解释修复现象。此为描述结论，不证明广度能预测之后收益；原模型也不因此被判定存在该字段误用。

决定：descriptive_breadth_price_and_basket_changes_separated。输出明确的price_component、basket_component及未知掩码供后续机制核查；不把分解本身当新alpha，不调整窗口，不修改现役模型。此前广度失败证据继续保留，稳定净超额仍未实现。

[全区间及季度汇总](decomposition_summary.csv) · [全部披露更新事件](disclosure_events.csv)

Recorder `079f5379920c4ac1aa295122f0f89789`。
